In [1]:
import io.github.francescodonnini.csv.CsvJavaClassApi
import io.github.francescodonnini.csv.CsvJavaMethodApi

val project = "SYNCOPE"
val path = "/home/francesco/Documents/Università/ISW/Progetto/60/data/$project"
val classApi = CsvJavaClassApi(path + "/classes.csv")
val methodApi = CsvJavaMethodApi(path + "/methods.csv", classApi.getLocal())
val methods = methodApi.getLocal()

In [2]:
import io.github.francescodonnini.csv.CsvReleaseApi

val releaseApi = CsvReleaseApi(path + "/releases.csv")
val releases = releaseApi.getLocal()
val lastRelease = releases.size/3

In [7]:
import io.github.francescodonnini.model.JavaMethod
import java.time.LocalDate

fun isBetween(m: JavaMethod, start: LocalDate, end: LocalDate) =
    !m.javaClass.time.toLocalDate().isBefore(start) && !m.javaClass.time.toLocalDate().isAfter(end)

fun key(m: JavaMethod) = "${m.javaClass.realPath}#${m.signature}"

fun printDiff(m: JavaMethod): String {
    val s = StringBuilder()
    s.append("DiffMetrics{")
    s.append("elseAdded=").append(m.metrics.elseAdded).append(", ")
    s.append("elseDeleted=").append(m.metrics.elseDeleted).append(", ")
    s.append("locAdded=").append(m.metrics.locAdded).append(", ")
    s.append("locDeleted=").append(m.metrics.locDeleted).append(", ")
    s.append("churn=").append(m.metrics.churn).append(", ")
    s.append("statementAdded=").append(m.metrics.statementsAdded).append(", ")
    s.append("statementDeleted=").append(m.metrics.statementsDeleted).append("}")
    return s.toString()
}

fun diff(revisions: List<JavaMethod>): JavaMethod? {
    if (revisions.isEmpty()) {
        return null
    } else if (revisions.size == 1) {
        return revisions[0]
    } else {
        val last = revisions.last()
        for (m in revisions.slice(0..revisions.size - 1)) {
            last.metrics.addLoc(m.metrics.lineOfCode)
            last.metrics.addElseCount(m.metrics.elseCount)
            last.metrics.addStatementCount(m.metrics.statementsCount)
            last.metrics.authors.forEach { a -> last.metrics.addAuthor(a) }
        }
        return last
    }
}

In [6]:
val start = LocalDate.MIN
val end = releases[lastRelease].releaseDate
val mapping = methods.filter { m -> isBetween(m, start, end) }.groupBy { m -> key(m) }

In [8]:
val result = mutableListOf<JavaMethod>()
var start = releases[0]
for (end in releases.slice(1..lastRelease)) {
    for ((k, revisions) in mapping) {
        revisions
            .filter { r -> isBetween(r, start.releaseDate, end.releaseDate) }
            .let { diff(it)?.let { result.add(it)} }
    }
}

In [11]:
 for (i in 0..10) {
     val m = result.random()
     println("${m.javaClass.realPath} ${m.javaClass.commit}")
     println("${m.signature} ${m.javaClass.time}")
     println(key(m))
     println(m.metrics)
     println(printDiff(m))
}

/home/francesco/Documents/Università/ISW/Progetto/60/sources/syncope/core/persistence-jpa/src/main/java/org/apache/syncope/core/persistence/jpa/entity/AbstractVirSchema.java 2d19463622aa937832b5deca845606c75898b9fd
void setReadonly(boolean) 2015-02-13T12:43:53
/home/francesco/Documents/Università/ISW/Progetto/60/sources/syncope/core/persistence-jpa/src/main/java/org/apache/syncope/core/persistence/jpa/entity/AbstractVirSchema.java#void setReadonly(boolean)
Metrics{cyclomaticComplexity=1, lineOfCode=4, parametersCount=1, statementsCount=1, elseCount=0, nestingDepth=0, authorsCount=0, authors=}
DiffMetrics{elseAdded=0, elseDeleted=0, locAdded=0, locDeleted=0, churn=0, statementAdded=0, statementDeleted=0}
/home/francesco/Documents/Università/ISW/Progetto/60/sources/syncope/core/persistence-jpa/src/main/java/org/apache/syncope/core/persistence/jpa/entity/role/JPARole.java 2d19463622aa937832b5deca845606c75898b9fd
List<? extends RVirAttr> getVirAttrs() 2015-02-13T12:43:53
/home/francesco/Do

In [13]:
mapping["/home/francesco/Documents/Università/ISW/Progetto/60/sources/syncope/core/src/main/java/org/apache/syncope/core/workflow/AbstractUserWorkflowAdapter.java#WorkflowResult<Long> suspend(SyncopeUser)"]?.forEach { println(it.metrics) }

Metrics{cyclomaticComplexity=1, lineOfCode=7, parametersCount=1, statementsCount=2, elseCount=0, nestingDepth=0, authorsCount=0, authors=}
Metrics{cyclomaticComplexity=1, lineOfCode=7, parametersCount=1, statementsCount=2, elseCount=0, nestingDepth=0, authorsCount=0, authors=}
Metrics{cyclomaticComplexity=1, lineOfCode=7, parametersCount=1, statementsCount=2, elseCount=0, nestingDepth=0, authorsCount=0, authors=}
